<a href="https://colab.research.google.com/github/oobayoshito/SD3A_2501902_yoshitoohba_mobile/blob/main/SD4AI%E6%BC%94%E7%BF%9207_02%E4%B9%85%E5%AE%B6%E6%94%BF%E4%BA%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 必要なライブラリのインストール（少し時間がかかります）
!pip install -q -U langchain-google-genai langchain pydantic

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# 鍵マークに登録したAPIキーを取得して設定
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# AIの脳みそ（モデル）を準備
# 今回は素早くて賢い「gemini-flash-latest」を使います
# 利用可能なモデルリストから選択
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.7)

print("AIの準備が完了しました！")

AIの準備が完了しました！


In [ ]:
from pydantic import BaseModel, Field
from typing import List

# AIに返してほしいデータの形（設計図）を作ります
class Recipe(BaseModel):
    # 材料は文字列のリスト（配列）で受け取る
    ingredients: List[str] = Field(description="料理の材料リスト")
    # 手順も文字列のリスト（配列）で受け取る
    steps: List[str] = Field(description="調理手順のリスト")

# 設計図をAIモデルにセット！ (これで決まった形で出力するようになります)
structured_llm = model.with_structured_output(Recipe)

print("データの設計図をAIにセットしました！")

データの設計図をAIにセットしました！


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# プロンプト（指示書）のテンプレートを作成
prompt = ChatPromptTemplate.from_messages([
    ("system", "ユーザーが入力した料理のレシピを考えてください。"),
    ("human", "{dish}"), # {dish} の部分に、後で好きな料理名が入ります
])

print("プロンプトの準備が完了しました！")

プロンプトの準備が完了しました！


In [ ]:
# プロンプトとモデルを「| (パイプ)」でつなぎます
chain = prompt | structured_llm

# {dish} の部分に「カレー」を入れてAIにお願いしてみる
print("AIがレシピを考え中です...")
recipe_object = chain.invoke({"dish": "カレー"})

# 結果の表示
print("\n=== AIからの回答 ===")
print("データの種類:", type(recipe_object))

print("\n【材料】")
for item in recipe_object.ingredients:
    print("・", item)

print("\n【手順】")
for i, step in enumerate(recipe_object.steps, 1):
    print(f"{i}. {step}")

AIがレシピを考え中です...

=== AIからの回答 ===
データの種類: <class '__main__.Recipe'>

【材料】
・ 豚肉 300g
・ 玉ねぎ 2個
・ 人参 1本
・ じゃがいも 2個
・ カレールー 1箱
・ 水 800ml
・ サラダ油 大さじ1

【手順】
1. 具材（肉、玉ねぎ、人参、じゃがいも）を一口大に切ります。
2. 鍋にサラダ油を熱し、切った具材を炒めます。
3. 玉ねぎがしんなりしたら水を加え、沸騰したらアクを取り、弱火で約15分煮込みます。
4. 火を止め、カレールーを割り入れてよく溶かします。
5. 再び弱火にかけ、時々混ぜながらとろみがつくまで約10分煮込みます。


In [ ]:
import google.generativeai as genai

for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

このリストから利用可能なモデル名を確認し、`model = ChatGoogleGenerativeAI(model="gemini-1.5-flash-8b", temperature=0.7)` の部分を修正してください。例えば、`gemini-pro` が利用可能であれば、`model="gemini-pro"` に変更します。